# Part 2: Web Search - Inverted Index
## CSL7110 - Machine Learning with Big Data | Assignment 4
**Name:** Aniket Srivastava | **Roll No:** M25DE1051

Implements a full inverted index pipeline:
- MySet, Position, WordEntry, PageIndex, MyHashTable, PageEntry
- InvertedPageIndex, SearchEngine
- TFIDF scoring
- Validated against answers.txt

In [ ]:
import os, math, re

# Stop words as defined in assignment
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are',
    'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

PUNCTUATION = set('{}[]<>=(). ,;\'"?#!-:')

# Plural normalization map (exhaustive per assignment)
PLURAL_MAP = {
    'stacks': 'stack', 'structures': 'structure', 'applications': 'application',
    'elements': 'element', 'operations': 'operation', 'implementations': 'implementation',
    'items': 'item', 'collections': 'collection', 'functions': 'function',
    'pages': 'page', 'words': 'word', 'indices': 'index', 'entries': 'entry',
    'engineers': 'engineer', 'plates': 'plate', 'terms': 'term',
    'webpages': 'webpage', 'queries': 'query', 'lists': 'list',
    'pushes': 'push', 'pops': 'pop', 'magazines': 'magazine',
}

def normalize(word):
    w = word.lower()
    return PLURAL_MAP.get(w, w)

def tokenize(text):
    for ch in PUNCTUATION:
        text = text.replace(ch, ' ')
    raw_tokens = text.split()
    result = []
    pos = 0
    for raw in raw_tokens:
        pos += 1
        w = normalize(raw)
        if w and w not in STOP_WORDS:
            result.append((pos, w))
    return result, pos

print("Utilities defined")

Utilities defined


## Classes: Position, WordEntry, MySet

In [ ]:
class Position:
    def __init__(self, page_entry, word_index):
        self._page = page_entry
        self._word_index = word_index
    def getPageEntry(self): return self._page
    def getWordIndex(self): return self._word_index

class WordEntry:
    def __init__(self, word):
        self.word = word
        self._positions = []
    def addPosition(self, pos): self._positions.append(pos)
    def addPositions(self, positions): self._positions.extend(positions)
    def getAllPositionsForThisWord(self): return list(self._positions)
    def getTermFrequency(self, page_entry):
        count = sum(1 for p in self._positions if p.getPageEntry() is page_entry)
        return count / page_entry.totalWords if page_entry.totalWords > 0 else 0.0

class MySet:
    def __init__(self): self._items = []
    def addElement(self, e):
        if e not in self._items: self._items.append(e)
    def union(self, other):
        r = MySet()
        for i in self._items: r.addElement(i)
        for i in other._items: r.addElement(i)
        return r
    def intersection(self, other):
        r = MySet()
        for i in self._items:
            if i in other._items: r.addElement(i)
        return r
    def toList(self): return list(self._items)
    def __len__(self): return len(self._items)

print("Position, WordEntry, MySet defined")

Position, WordEntry, MySet defined


## Classes: PageIndex, MyHashTable, PageEntry

In [ ]:
class PageIndex:
    def __init__(self): self._word_entries = {}
    def addPositionForWord(self, word, position):
        if word not in self._word_entries:
            self._word_entries[word] = WordEntry(word)
        self._word_entries[word].addPosition(position)
    def getWordEntries(self): return list(self._word_entries.values())
    def containsWord(self, word): return word in self._word_entries
    def getPositions(self, word):
        we = self._word_entries.get(word)
        return we.getAllPositionsForThisWord() if we else []

class MyHashTable:
    def __init__(self, size=1024):
        self._size = size
        self._table = [None] * size
    def getHashIndex(self, word):
        h = 0
        for ch in word: h = (h * 31 + ord(ch)) % self._size
        return h
    def addPositionsForWord(self, word_entry):
        idx = self.getHashIndex(word_entry.word)
        if self._table[idx] is None: self._table[idx] = {}
        if word_entry.word in self._table[idx]:
            self._table[idx][word_entry.word].addPositions(word_entry.getAllPositionsForThisWord())
        else:
            self._table[idx][word_entry.word] = word_entry
    def getWordEntry(self, word):
        idx = self.getHashIndex(word)
        if self._table[idx] is None: return None
        return self._table[idx].get(word, None)

class PageEntry:
    def __init__(self, page_name, webpages_dir):
        self.pageName = page_name
        filepath = os.path.join(webpages_dir, page_name)
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        tokens, self.totalWords = tokenize(text)
        self._page_index = PageIndex()
        for pos, word in tokens:
            self._page_index.addPositionForWord(word, Position(self, pos))
    def getPageIndex(self): return self._page_index

print("PageIndex, MyHashTable, PageEntry defined")

PageIndex, MyHashTable, PageEntry defined


## Classes: InvertedPageIndex, SearchEngine

In [ ]:
class InvertedPageIndex:
    def __init__(self):
        self._pages = {}
        self._hash_table = MyHashTable()
        self._total_pages = 0
    def addPage(self, page_entry):
        self._pages[page_entry.pageName] = page_entry
        self._total_pages += 1
        for we in page_entry.getPageIndex().getWordEntries():
            self._hash_table.addPositionsForWord(we)
    def getPagesWhichContainWord(self, word):
        nw = normalize(word)
        result = MySet()
        for page in self._pages.values():
            if page.getPageIndex().containsWord(nw):
                result.addElement(page)
        return result
    def getPage(self, name): return self._pages.get(name, None)
    def getAllPages(self): return list(self._pages.values())
    def getTotalPages(self): return self._total_pages

class SearchEngine:
    def __init__(self, webpages_dir):
        self._index = InvertedPageIndex()
        self._webpages_dir = webpages_dir
    def performAction(self, action_message):
        parts = action_message.strip().split()
        if not parts: return ""
        action = parts[0]
        if action == "addPage" and len(parts) >= 2:
            pe = PageEntry(parts[1], self._webpages_dir)
            self._index.addPage(pe)
            return f"Added page: {parts[1]}"
        elif action == "queryFindPagesWhichContainWord" and len(parts) >= 2:
            word = parts[1]; nw = normalize(word)
            pages = self._index.getPagesWhichContainWord(nw)
            pl = pages.toList()
            if not pl: return f"No webpage contains word {word}"
            return ", ".join(sorted([p.pageName for p in pl]))
        elif action == "queryFindPositionsOfWordInAPage" and len(parts) >= 3:
            word, page_name = parts[1], parts[2]; nw = normalize(word)
            page = self._index.getPage(page_name)
            if page is None: return f"No webpage {page_name} found"
            positions = page.getPageIndex().getPositions(nw)
            if not positions: return f"Webpage {page_name} does not contain word {word}"
            return ", ".join(map(str, sorted([p.getWordIndex() for p in positions])))
        return f"Unknown action: {action}"

print("InvertedPageIndex, SearchEngine defined")

InvertedPageIndex, SearchEngine defined


## Process actions.txt and Validate

In [ ]:
WEBPAGES_DIR = '../data/Q2/webpages'
ACTIONS_FILE = '../data/Q2/actions.txt'
ANSWERS_FILE = '../data/Q2/answers.txt'

engine = SearchEngine(WEBPAGES_DIR)

with open(ACTIONS_FILE) as f:
    actions = [l.strip() for l in f if l.strip()]
with open(ANSWERS_FILE) as f:
    answers = [l.strip() for l in f if l.strip()]

query_outputs = []
print("Processing actions:\n")
for action in actions:
    result = engine.performAction(action)
    print(f"  > {action}")
    if not action.startswith("addPage"):
        print(f"    Output : {result}")
        query_outputs.append(result)

print("\nValidation:")
all_ok = True
for i, (out, exp) in enumerate(zip(query_outputs, answers)):
    match = out.strip() == exp.strip()
    if not match: all_ok = False
    print(f"  Q{i+1}: [{"PASS" if match else "FAIL"}]")

print("\nAll correct!" if all_ok else "\nSome mismatches.")

Processing actions:

  > addPage stack_datastructure_wiki
  > queryFindPagesWhichContainWord delhi
    Output : No webpage contains word delhi
  > queryFindPagesWhichContainWord stack
    Output : stack_datastructure_wiki
  > queryFindPagesWhichContainWord wikipedia
    Output : stack_datastructure_wiki
  > queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
    Output : Webpage stack_datastructure_wiki does not contain word magazines
  > queryFindPagesWhichContainWord allain
    Output : No webpage contains word allain
  > addPage stack_cprogramming
  > queryFindPagesWhichContainWord allain
    Output : stack_cprogramming
  > queryFindPagesWhichContainWord C
    Output : stack_cprogramming
  > queryFindPagesWhichContainWord C++
    Output : stack_cprogramming
  > addPage stack_oracle
  > queryFindPagesWhichContainWord jdk
    Output : stack_oracle
  > addPage stackoverflow
  > queryFindPagesWhichContainWord function
    Output : stack_cprogramming, stack_datastructure_w